In [21]:
import os
import re
import time
import json
from pathlib import Path
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass

import pdfplumber
from tqdm import tqdm
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

ENV_PATH: str = '/home/akel/PycharmProjects/InsurMinds2026/.env'

def carrega_variaveis_ambiente() -> None:
    if os.path.exists(ENV_PATH):
        load_dotenv(ENV_PATH, override=True)
        print("✔ Variáveis de ambiente carregadas do arquivo .env")
    else:
        print(f"⚠ Aviso: Arquivo {ENV_PATH} não foi encontrado no diretório atual.")

print('✔ OUTPUT_DOCUMENTS_DIR:',OUTPUT_DOCUMENTS_DIR)
carrega_variaveis_ambiente()

print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


✔ OUTPUT_DOCUMENTS_DIR: data/fonte/
✔ Variáveis de ambiente carregadas do arquivo .env
# 12/08/2026 - 14:24:48


In [22]:
# ==========================================
# CONFIGURAÇÃO
# ==========================================
OUTPUT_DOCUMENTS_DIR = "data/fonte/"
VECTORSTORE_DIR = "data/vectorstore_teste"

# ==========================================
# ESTRUTURAS DE DADOS
# ==========================================
@dataclass
class TabelaExtraida:
    """Representa uma tabela extraída do PDF."""
    pagina: int
    bbox: Tuple[float, float, float, float]
    dados: List[List[str]]
    texto_original: str
    markdown: str
    
@dataclass
class PaginaProcessada:
    """Representa uma página processada do PDF."""
    numero: int
    texto_sem_tabelas: str
    tabelas: List[TabelaExtraida]
    texto_completo: str
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


# 12/08/2026 - 14:24:52


In [23]:
# ==========================================
# 1. EXTRAÇÃO COM PDFPLUMBER
# ==========================================
class ExtratorPDFAvancado:
    """
    Extrator de PDF especializado em separar tabelas do texto.
    Usa pdfplumber para detecção precisa de tabelas.
    """
    
    def __init__(self, diretorio_pdf: str):
        self.diretorio_pdf = diretorio_pdf
        self.paginas_processadas = []
        
    def extrair_documento(self) -> List[PaginaProcessada]:
        """
        Processa todos os PDFs do diretório.
        """
        print(f"\n📂 Procurando PDFs em: {self.diretorio_pdf}")
        pdfs = list(Path(self.diretorio_pdf).glob("*.pdf"))
        
        if not pdfs:
            raise ValueError(f"Nenhum PDF encontrado em {self.diretorio_pdf}")
        
        print(f"✅ {len(pdfs)} PDFs encontrados")
        
        for pdf_path in pdfs:
            print(f"\n📄 Processando: {pdf_path.name}")
            self.processar_pdf(pdf_path)
        
        return self.paginas_processadas
    
    def processar_pdf(self, pdf_path: Path):
        """
        Processa um único PDF.
        """
        try:
            with pdfplumber.open(pdf_path) as pdf:
                total_paginas = len(pdf.pages)
                
                for num_pagina, pagina in enumerate(tqdm(pdf.pages, desc="Extraindo páginas")):
                    pagina_processada = self.processar_pagina(
                        pagina, 
                        num_pagina, 
                        pdf_path.name
                    )
                    self.paginas_processadas.append(pagina_processada)
                    
        except Exception as e:
            print(f"❌ Erro ao processar {pdf_path.name}: {str(e)}")
    
    def processar_pagina(
        self, 
        pagina: pdfplumber.page.Page, 
        num_pagina: int, 
        fonte: str
    ) -> PaginaProcessada:
        """
        Processa uma página individual, separando tabelas do texto.
        """
        # 1. Extrai tabelas
        tabelas_extraidas = self.extrair_tabelas(pagina, num_pagina)
        
        # 2. Extrai texto
        texto_completo = pagina.extract_text() or ""
        
        # 3. Remove tabelas do texto (para não duplicar)
        texto_sem_tabelas = self.remover_tabelas_do_texto(
            texto_completo, 
            tabelas_extraidas
        )
        
        # 4. Cria objeto da página processada
        pagina_processada = PaginaProcessada(
            numero=num_pagina,
            texto_sem_tabelas=texto_sem_tabelas,
            tabelas=tabelas_extraidas,
            texto_completo=texto_completo
        )
        
        # Log
        if tabelas_extraidas:
            print(f"   📊 {len(tabelas_extraidas)} tabela(s) encontrada(s) na página {num_pagina + 1}")
        
        return pagina_processada
    
    def extrair_tabelas(
        self, 
        pagina: pdfplumber.page.Page, 
        num_pagina: int
    ) -> List[TabelaExtraida]:
        """
        Extrai todas as tabelas de uma página usando pdfplumber.
        """
        tabelas_extraidas = []
        
        try:
            # Estratégia 1: Usa configurações de tabela do pdfplumber
            tabelas = pagina.extract_tables({
                "vertical_strategy": "lines",
                "horizontal_strategy": "lines",
                "snap_tolerance": 3,
                "join_tolerance": 3,
                "edge_min_length": 3,
                "min_words_vertical": 2,
                "min_words_horizontal": 1,
                "intersection_tolerance": 3,
            })
            
            for i, tabela in enumerate(tabelas):
                if tabela and len(tabela) > 1:  # Mínimo 2 linhas
                    # Limpa células
                    tabela_limpa = self.limpar_tabela(tabela)
                    
                    # Converte para Markdown
                    markdown = self.tabela_para_markdown(tabela_limpa)
                    
                    # Cria objeto TabelaExtraida
                    tabela_obj = TabelaExtraida(
                        pagina=num_pagina + 1,
                        bbox=(0, 0, 0, 0),  # pdfplumber não fornece bbox diretamente
                        dados=tabela_limpa,
                        texto_original=str(tabela),
                        markdown=markdown
                    )
                    
                    tabelas_extraidas.append(tabela_obj)
            
            # Estratégia 2: Se não encontrou tabelas, tenta abordagem alternativa
            if not tabelas_extraidas:
                tabelas_alternativas = self.extrair_tabelas_alternativo(pagina)
                tabelas_extraidas.extend(tabelas_alternativas)
        
        except Exception as e:
            print(f"   ⚠️  Erro ao extrair tabelas da página {num_pagina + 1}: {str(e)}")
        
        return tabelas_extraidas
    
    def extrair_tabelas_alternativo(
        self, 
        pagina: pdfplumber.page.Page
    ) -> List[TabelaExtraida]:
        """
        Método alternativo para detectar tabelas quando o método principal falha.
        Usa heurísticas baseadas em alinhamento de palavras.
        """
        tabelas = []
        
        try:
            # Extrai palavras com posições
            palavras = pagina.extract_words(
                keep_blank_chars=True,
                use_text_flow=False,
                extra_attrs=['size', 'fontname']
            )
            
            if not palavras:
                return tabelas
            
            # Agrupa palavras por linha (baseado na posição y)
            linhas = {}
            for palavra in palavras:
                y = round(palavra['top'] / 5) * 5  # Agrupa por proximidade vertical
                if y not in linhas:
                    linhas[y] = []
                linhas[y].append(palavra)
            
            # Ordena linhas por posição
            linhas_ordenadas = sorted(linhas.items())
            
            # Detecta padrões de tabela (múltiplas colunas alinhadas)
            for y, palavras_linha in linhas_ordenadas:
                if len(palavras_linha) >= 3:  # Mínimo 3 palavras para considerar tabela
                    # Verifica alinhamento horizontal
                    x_positions = [p['x0'] for p in palavras_linha]
                    x_positions.sort()
                    
                    # Verifica se há espaçamento regular (indicativo de colunas)
                    espacamentos = [
                        x_positions[i+1] - x_positions[i] 
                        for i in range(len(x_positions)-1)
                    ]
                    
                    if any(esp > 20 for esp in espacamentos):  # Espaço > 20pt
                        # Pode ser uma tabela
                        tabela_linhas = []
                        for linha_y, linha_palavras in linhas_ordenadas:
                            if linha_y >= y - 10 and linha_y <= y + 100:
                                texto_linha = ' | '.join(
                                    p['text'] for p in sorted(
                                        linha_palavras, 
                                        key=lambda x: x['x0']
                                    )
                                )
                                tabela_linhas.append(texto_linha)
                        
                        if len(tabela_linhas) >= 2:
                            markdown = self.texto_para_markdown_tabela(tabela_linhas)
                            tabela_obj = TabelaExtraida(
                                pagina=0,
                                bbox=(0, 0, 0, 0),
                                dados=[],
                                texto_original='\n'.join(tabela_linhas),
                                markdown=markdown
                            )
                            tabelas.append(tabela_obj)
                            break
        
        except Exception as e:
            pass
        
        return tabelas
    
    def limpar_tabela(self, tabela: List[List]) -> List[List[str]]:
        """
        Limpa e normaliza células da tabela.
        """
        tabela_limpa = []
        
        for linha in tabela:
            linha_limpa = []
            for celula in linha:
                if celula is None:
                    celula = ""
                # Remove quebras de linha e espaços extras
                celula = re.sub(r'\s+', ' ', str(celula)).strip()
                linha_limpa.append(celula)
            
            # Remove linhas completamente vazias
            if any(linha_limpa):
                tabela_limpa.append(linha_limpa)
        
        return tabela_limpa
    
    def tabela_para_markdown(self, tabela: List[List[str]]) -> str:
        """
        Converte tabela para formato Markdown.
        """
        if not tabela:
            return ""
        
        # Determina número máximo de colunas
        max_colunas = max(len(linha) for linha in tabela)
        
        # Normaliza número de colunas
        tabela_normalizada = []
        for linha in tabela:
            while len(linha) < max_colunas:
                linha.append("")
            tabela_normalizada.append(linha)
        
        # Cria Markdown
        cabecalho = tabela_normalizada[0]
        separator = ['---'] * max_colunas
        
        markdown_lines = []
        markdown_lines.append("| " + " | ".join(cabecalho) + " |")
        markdown_lines.append("| " + " | ".join(separator) + " |")
        
        for linha in tabela_normalizada[1:]:
            markdown_lines.append("| " + " | ".join(linha) + " |")
        
        return "\n".join(markdown_lines)
    
    def texto_para_markdown_tabela(self, linhas: List[str]) -> str:
        """
        Converte linhas de texto em tabela Markdown.
        """
        if not linhas:
            return ""
        
        # Divide linhas em colunas
        dados = []
        for linha in linhas:
            colunas = [c.strip() for c in linha.split('|')]
            dados.append(colunas)
        
        return self.tabela_para_markdown(dados)
    
    def remover_tabelas_do_texto(
        self, 
        texto: str, 
        tabelas: List[TabelaExtraida]
    ) -> str:
        """
        Remove o conteúdo das tabelas do texto corrido para evitar duplicação.
        """
        if not tabelas:
            return texto
        
        texto_limpo = texto
        
        for tabela in tabelas:
            # Remove linhas que pertencem à tabela
            for linha in tabela.texto_original.split('\n'):
                if linha.strip():
                    texto_limpo = texto_limpo.replace(linha.strip(), '')
        
        # Limpa linhas vazias extras
        texto_limpo = re.sub(r'\n{3,}', '\n\n', texto_limpo)
        
        return texto_limpo.strip()

# ==========================================
# 2. PROCESSAMENTO DE CHUNKS
# ==========================================
class ProcessadorChunks:
    """
    Processa páginas extraídas em chunks otimizados.
    """
    
    def __init__(self):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=200,
            separators=["\n\n", "\n", ". ", " ", ""],
            length_function=len,
        )
    
    def criar_chunks(self, paginas: List[PaginaProcessada]) -> List[Document]:
        """
        Cria chunks separando texto e tabelas.
        """
        chunks = []
        chunk_id = 0
        
        for pagina in paginas:
            # 1. Chunks de texto (sem tabelas)
            if pagina.texto_sem_tabelas.strip():
                textos_divididos = self.text_splitter.split_text(
                    pagina.texto_sem_tabelas
                )
                
                for texto in textos_divididos:
                    if texto.strip():
                        chunk = Document(
                            page_content=texto,
                            metadata={
                                "tipo": "texto",  # string
                                "pagina": pagina.numero + 1,  # int
                                "chunk_id": chunk_id,  # int
                                "tem_tabela": False,  # boolean
                                "num_tabelas": 0,  # int
                                "fonte": "relatorio_seguranca",  # string
                            }
                        )
                        chunks.append(chunk)
                        chunk_id += 1
            
            # 2. Chunks de tabelas (cada tabela vira um chunk)
            for tabela in pagina.tabelas:
                # Adiciona contexto da página
                contexto = self.extrair_contexto_tabela(
                    pagina, 
                    tabela
                )
                
                conteudo_final = f"{contexto}\n\n{tabela.markdown}"
                
                # Determina número de linhas e colunas
                num_linhas = len(tabela.dados) if tabela.dados else 0
                num_colunas = len(tabela.dados[0]) if tabela.dados else 0
                
                chunk = Document(
                    page_content=conteudo_final,
                    metadata={
                        "tipo": "tabela",  # string
                        "pagina": tabela.pagina,  # int
                        "chunk_id": chunk_id,  # int
                        "tem_tabela": True,  # boolean
                        "num_tabelas": 1,  # int
                        "num_linhas": num_linhas,  # int
                        "num_colunas": num_colunas,  # int
                        "fonte": "relatorio_seguranca",  # string
                    }
                )
                chunks.append(chunk)
                chunk_id += 1
        
        # Adiciona validação final dos metadados
        chunks_validados = self.validar_metadados(chunks)
        
        return chunks_validados


    def validar_metadados(self, chunks: List[Document]) -> List[Document]:
        """
        Garante que todos os metadados são compatíveis com ChromaDB.
        """
        for chunk in chunks:
            metadados_limpos = {}
            
            for chave, valor in chunk.metadata.items():
                # Converte tipos não suportados
                if isinstance(valor, list):
                    # Converte lista para string
                    if len(valor) > 0:
                        metadados_limpos[chave] = ", ".join(str(v) for v in valor)
                    else:
                        metadados_limpos[chave] = None  # ou "vazio"
                elif isinstance(valor, (str, int, float, bool)):
                    # Tipos aceitos pelo ChromaDB
                    metadados_limpos[chave] = valor
                elif valor is None:
                    # None é aceito
                    metadados_limpos[chave] = None
                else:
                    # Converte outros tipos para string
                    metadados_limpos[chave] = str(valor)
            
            # Atualiza com metadados validados
            chunk.metadata = metadados_limpos
        
        return chunks
    
    def extrair_contexto_tabela(
        self, 
        pagina: PaginaProcessada, 
        tabela: TabelaExtraida
    ) -> str:
        """
        Extrai contexto ao redor da tabela para melhor compreensão.
        """
        contexto = f"📊 TABELA DA PÁGINA {tabela.pagina}\n"
        
        # Tenta encontrar título ou descrição próxima
        linhas_texto = pagina.texto_completo.split('\n')
        
        for i, linha in enumerate(linhas_texto):
            if linha.strip() and len(linha.strip()) < 100:
                # Verifica se parece um título (antes da tabela)
                if any(palavra in linha.lower() for palavra in [
                    'tabela', 'quadro', 'gráfico', 'estatística', 'dados',
                    'evolução', 'comparativo', 'indicador', 'taxa', 'índice'
                ]):
                    contexto += f"\n📋 {linha.strip()}\n"
                    break
        
        return contexto

# ==========================================
# 3. ENRIQUECIMENTO DE METADADOS
# ==========================================
class EnriquecerMetadados:
    """
    Adiciona metadados específicos para segurança pública.
    Compatível com ChromaDB (sem listas vazias).
    """
    
    def __init__(self):
        self.padroes = {
            'homicidios': [
                r'homicídio', r'assassinato', r'morte violenta',
                r'crime contra a vida', r'latrocínio'
            ],
            'crimes_patrimoniais': [
                r'roubo', r'furto', r'assalto', r'crime patrimonial',
                r'extorsão'
            ],
            'narcoticos': [
                r'tráfico', r'drogas', r'entorpecentes', r'apreensão',
                r'narcóticos'
            ],
            'violencia_genero': [
                r'feminicídio', r'violência doméstica', r'estupro',
                r'violência contra a mulher'
            ],
            'armas': [
                r'arma', r'fogo', r'munição', r'apreensão de armas',
                r'porte ilegal'
            ],
            'crimes_ciberneticos': [
                r'cibernético', r'estelionato', r'fraude', r'virtual',
                r'internet'
            ],
        }
    
    def enriquecer(self, chunks: List[Document]) -> List[Document]:
        """
        Enriquece chunks com metadados compatíveis com ChromaDB.
        """
        for chunk in chunks:
            texto = chunk.page_content.lower()
            
            # Detecta categorias (substitui lista vazia por string "nenhuma")
            categorias = []
            for categoria, padroes in self.padroes.items():
                if any(re.search(padrao, texto, re.IGNORECASE) for padrao in padroes):
                    categorias.append(categoria)
            
            # CORREÇÃO: Converte lista vazia para string "nenhuma"
            categoria_principal = categorias[0] if categorias else "geral"
            categorias_str = ", ".join(categorias) if categorias else "geral"
            
            # Detecta dados numéricos
            numeros = re.findall(r'\d+[\.,]?\d*', chunk.page_content)
            
            # Detecta períodos temporais
            anos = re.findall(r'\b(19|20)\d{2}\b', texto)
            meses = re.findall(
                r'\b(janeiro|fevereiro|março|abril|maio|junho|'
                r'julho|agosto|setembro|outubro|novembro|dezembro)\b',
                texto,
                re.IGNORECASE
            )
            
            # Converte listas para strings ou None (evita listas vazias)
            anos_str = ", ".join(sorted(set(anos))) if anos else None
            meses_str = ", ".join(sorted(set(meses))) if meses else None
            
            # Atualiza metadados (apenas tipos aceitos pelo ChromaDB)
            chunk.metadata.update({
                "categoria_principal": categoria_principal,  # string
                "categorias": categorias_str,  # string, não lista
                "tem_dados_numericos": len(numeros) > 5,  # boolean
                "quantidade_numeros": len(numeros),  # int
                "anos_referencia": anos_str,  # string ou None
                "meses_referencia": meses_str,  # string ou None
                "densidade_informacao": round(len(numeros) / max(len(texto), 1), 4),  # float
            })
        
        return chunks
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


# 12/08/2026 - 14:24:56


In [24]:
# # 1. Extração com pdfplumber
print("\n📄 ETAPA 1: Extração de PDFs")
print("-" * 40)
extrator = ExtratorPDFAvancado(OUTPUT_DOCUMENTS_DIR)
paginas = extrator.extrair_documento()
    
# Estatísticas
total_tabelas = sum(len(p.tabelas) for p in paginas)
total_paginas = len(paginas)
    
print(f"\n📊 Resumo da Extração:")
print(f"   Total de páginas: {total_paginas}")
print(f"   Total de tabelas: {total_tabelas}")
print(f"   Média de tabelas por página: {total_tabelas/total_paginas:.2f}")


📄 ETAPA 1: Extração de PDFs
----------------------------------------

📂 Procurando PDFs em: data/fonte/
✅ 1 PDFs encontrados

📄 Processando: anuario_2026.pdf


Extraindo páginas:   0%|                       | 1/424 [00:00<00:47,  8.90it/s]

   📊 1 tabela(s) encontrada(s) na página 1


Extraindo páginas:   1%|▏                      | 4/424 [00:00<00:37, 11.21it/s]

   📊 1 tabela(s) encontrada(s) na página 4
   📊 1 tabela(s) encontrada(s) na página 5
   📊 1 tabela(s) encontrada(s) na página 6


...

Extraindo páginas:  99%|████████████████████▊| 420/424 [01:14<00:00, 10.07it/s]

   📊 1 tabela(s) encontrada(s) na página 419
   📊 1 tabela(s) encontrada(s) na página 420
   📊 1 tabela(s) encontrada(s) na página 421


Extraindo páginas: 100%|█████████████████████| 424/424 [01:14<00:00,  5.69it/s]

   📊 1 tabela(s) encontrada(s) na página 422



📊 Resumo da Extração:
   Total de páginas: 424
   Total de tabelas: 581
   Média de tabelas por página: 1.37


In [6]:
# 2. Criação de chunks
print("\n📝 ETAPA 2: Criação de Chunks")
print("-" * 40)
processador = ProcessadorChunks()
chunks = processador.criar_chunks(paginas)
    
# Estatísticas de chunks
chunks_texto = sum(1 for c in chunks if c.metadata['tipo'] == 'texto')
chunks_tabela = sum(1 for c in chunks if c.metadata['tipo'] == 'tabela')
    
print(f"\n📊 Resumo dos Chunks:")
print(f"   Chunks de texto: {chunks_texto}")
print(f"   Chunks de tabela: {chunks_tabela}")
print(f"   Total: {len(chunks)}")



📝 ETAPA 2: Criação de Chunks
----------------------------------------

📊 Resumo dos Chunks:
   Chunks de texto: 1015
   Chunks de tabela: 581
   Total: 1596


In [7]:
# 3. Enriquecimento de metadados
print("\n🏷️  ETAPA 3: Enriquecimento de Metadados")
print("-" * 40)
enriquecedor = EnriquecerMetadados()
chunks = enriquecedor.enriquecer(chunks)
print(f"✅ {len(chunks)} chunks enriquecidos")



🏷️  ETAPA 3: Enriquecimento de Metadados
----------------------------------------
✅ 1596 chunks enriquecidos


In [10]:
# 4. Criação de embeddings


print("\n🧮 ETAPA 4: Configuração de Embeddings")
print("-" * 40)
embedding_model = HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-small',
    model_kwargs={
            "device": "cpu",
            "trust_remote_code": True
        },
        encode_kwargs={
            "normalize_embeddings": True,
        }
    )
print(f"✅ Modelo carregado: {embedding_model.model_name}")
    
    

✔ OUTPUT_DOCUMENTS_DIR: data/fonte/
✔ Variáveis de ambiente carregadas do arquivo .env

🧮 ETAPA 4: Configuração de Embeddings
----------------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Modelo carregado: intfloat/multilingual-e5-small


In [26]:
# ==========================================
# CELL 8 (VERSÃO FINAL) - Criação do vectorstore
# ==========================================
import os
import shutil
import stat
import gc
import time

print("\n💾 ETAPA 5: Criação do Banco Vetorial")
print("-" * 40)

# ==========================================
# 1. FECHAR INSTÂNCIAS ANTERIORES DO CHROMADB
# ==========================================
print("🔒 Fechando instâncias anteriores do ChromaDB...")

# Se existir uma variável 'vectorstore' de células anteriores, tenta fechar
try:
    if 'vectorstore' in globals() and vectorstore is not None:
        # Tenta fechar o cliente
        if hasattr(vectorstore, '_client'):
            vectorstore._client.clear_system_cache()
        del vectorstore
        print("✅ Instância anterior removida")
except Exception as e:
    print(f"⚠️ Não foi possível fechar instância anterior: {e}")

# Força coleta de lixo
gc.collect()
time.sleep(1)

# ==========================================
# 2. REMOÇÃO SEGURA DO DIRETÓRIO ANTIGO
# ==========================================
VECTORSTORE_DIR = "data/vectorstore_teste"

def tornar_escrevivel(func, path, exc_info):
    """Torna arquivos/diretórios escrevíveis antes de remover."""
    os.chmod(path, stat.S_IWRITE | stat.S_IREAD | stat.S_IEXEC)
    func(path)

if os.path.exists(VECTORSTORE_DIR):
    print(f"⚠️ Diretório existente: {VECTORSTORE_DIR}")
    print("🔧 Removendo com alteração de permissões...")
    try:
        shutil.rmtree(VECTORSTORE_DIR, onerror=tornar_escrevivel)
        print("✅ Diretório antigo removido com sucesso")
    except Exception as e:
        print(f"❌ Erro ao remover: {e}")
        print("🔄 Tentando remoção forçada arquivo por arquivo...")
        for root, dirs, files in os.walk(VECTORSTORE_DIR, topdown=False):
            for name in files:
                file_path = os.path.join(root, name)
                try:
                    os.chmod(file_path, stat.S_IWRITE)
                    os.remove(file_path)
                except:
                    pass
            for name in dirs:
                dir_path = os.path.join(root, name)
                try:
                    os.chmod(dir_path, stat.S_IWRITE | stat.S_IREAD | stat.S_IEXEC)
                    os.rmdir(dir_path)
                except:
                    pass
        try:
            os.rmdir(VECTORSTORE_DIR)
            print("✅ Diretório removido (forçado)")
        except:
            print("⚠️ Não foi possível remover completamente; usando diretório alternativo")
            VECTORSTORE_DIR = "data/vectorstore_teste_novo"

# ==========================================
# 3. CRIA NOVO DIRETÓRIO COM PERMISSÕES CORRETAS
# ==========================================
print(f"📁 Criando diretório: {VECTORSTORE_DIR}")
os.makedirs(VECTORSTORE_DIR, exist_ok=True)

# Concede permissões totais (rwx para todos)
os.chmod(VECTORSTORE_DIR, stat.S_IRWXU | stat.S_IRWXG | stat.S_IROTH | stat.S_IXOTH)
print("✅ Permissões definidas")

# ==========================================
# 4. LIMPEZA DE METADADOS (GARANTIA)
# ==========================================
def limpar_metadados_para_chromadb(chunks):
    chunks_limpos = []
    for chunk in chunks:
        metadados_novos = {}
        for chave, valor in chunk.metadata.items():
            if isinstance(valor, list) and len(valor) == 0:
                metadados_novos[chave] = None
            elif isinstance(valor, list):
                metadados_novos[chave] = ", ".join(str(v) for v in valor)
            elif isinstance(valor, (str, int, float, bool)) or valor is None:
                metadados_novos[chave] = valor
            elif isinstance(valor, dict):
                metadados_novos[chave] = json.dumps(valor, ensure_ascii=False)
            else:
                metadados_novos[chave] = str(valor)
        chunk_limpo = Document(page_content=chunk.page_content, metadata=metadados_novos)
        chunks_limpos.append(chunk_limpo)
    return chunks_limpos

print("🧹 Limpando metadados...")
chunks_limpos = limpar_metadados_para_chromadb(chunks)
print(f"✅ {len(chunks_limpos)} chunks prontos")

# ==========================================
# 5. CRIAÇÃO DO VECTORSTORE
# ==========================================
print(f"\n📦 Criando banco com {len(chunks_limpos)} chunks...")

try:
    vectorstore = Chroma.from_documents(
        documents=chunks_limpos[:batch_size],
        embedding=embedding_model,
        persist_directory=VECTORSTORE_DIR,
        collection_metadata={
            "hnsw:space": "cosine",
            "description": "Relatórios de Segurança Pública com tabelas em Markdown"
        }
    )
    
    # Adiciona lotes restantes
    for i in tqdm(range(batch_size, len(chunks_limpos), batch_size), desc="Indexando chunks"):
        lote = chunks_limpos[i:i + batch_size]
        vectorstore.add_documents(lote)
    
    print(f"\n✅ Banco criado com sucesso!")
    print(f"📊 Total de chunks indexados: {vectorstore._collection.count()}")

except InternalError as e:
    print(f"\n❌ Erro de banco readonly: {e}")
    print("🔧 Tentando com diretório alternativo...")
    
    # Usa um diretório totalmente novo com timestamp
    VECTORSTORE_DIR_NOVO = f"data/vectorstore_{int(time.time())}"
    print(f"📁 Novo diretório: {VECTORSTORE_DIR_NOVO}")
    
    os.makedirs(VECTORSTORE_DIR_NOVO, exist_ok=True)
    os.chmod(VECTORSTORE_DIR_NOVO, stat.S_IRWXU | stat.S_IRWXG | stat.S_IROTH | stat.S_IXOTH)
    
    try:
        vectorstore = Chroma.from_documents(
            documents=chunks_limpos[:batch_size],
            embedding=embedding_model,
            persist_directory=VECTORSTORE_DIR_NOVO,
            collection_metadata={"hnsw:space": "cosine"}
        )
        for i in tqdm(range(batch_size, len(chunks_limpos), batch_size), desc="Indexando chunks"):
            lote = chunks_limpos[i:i + batch_size]
            vectorstore.add_documents(lote)
        print(f"\n✅ Banco criado em diretório alternativo: {VECTORSTORE_DIR_NOVO}")
        print(f"📊 Total de chunks: {vectorstore._collection.count()}")
    except Exception as e2:
        print(f"\n❌ Falha também no diretório alternativo: {e2}")
        print("🚨 Possível causa: permissões do sistema de arquivos ou falta de espaço.")
        raise

except Exception as e:
    print(f"\n❌ Erro inesperado: {e}")
    raise


💾 ETAPA 5: Criação do Banco Vetorial
----------------------------------------
🔒 Fechando instâncias anteriores do ChromaDB...
📁 Criando diretório: data/vectorstore_teste
✅ Permissões definidas
🧹 Limpando metadados...
✅ 1596 chunks prontos

📦 Criando banco com 1596 chunks...


NameError: name 'InternalError' is not defined

In [ ]:
    # 6. Estatísticas finais
print("\n" + "=" * 80)
print("✅ INGESTÃO CONCLUÍDA COM SUCESSO")
print("=" * 80)
print(f"📁 Banco criado em: {VECTORSTORE_DIR}")
print(f"📊 Total de chunks: {vectorstore._collection.count()}")
print(f"📈 Tabelas preservadas: {chunks_tabela}")
print(f"🔢 Dados numéricos: {sum(1 for c in chunks if c.metadata['tem_dados_numericos'])}")
print("=" * 80)
    
# Salva estatísticas
estatisticas = {
        "total_paginas": total_paginas,
        "total_tabelas": total_tabelas,
        "total_chunks": len(chunks),
        "chunks_texto": chunks_texto,
        "chunks_tabela": chunks_tabela,
        "data_ingestao": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    
with open(f"{VECTORSTORE_DIR}/estatisticas.json", "w") as f:
        json.dump(estatisticas, f, indent=2, ensure_ascii=False)
    


